# Gradient-Descent Optimizers — momentum, adaptivity, and Adam

> Tutorial pair for [`optimizers.py`](optimizers.py).

## 1. Intuition
All these optimizers do the same thing — step downhill — but they disagree on
*how big* and *in which direction* the step should be. **Momentum** remembers
where it was going and damps zig-zags. **Adaptive** methods (AdaGrad/RMSProp/Adam)
give each parameter its own learning rate based on the history of its gradients.
**Adam** combines both and adds **bias correction** to fix the cold-start.

## 2. Concept (the slide)
On an **ill-conditioned** loss (long, narrow valley) plain SGD bounces across the
steep walls while crawling along the floor. The cures:
- **Momentum / Nesterov:** average gradients into a velocity → cancel the bounce,
  accelerate along the floor. Nesterov peeks at the *lookahead* point first.
- **AdaGrad:** divide by $\sqrt{\sum g^2}$ → big steps for rare directions; but the
  sum only grows, so the LR decays to zero.
- **RMSProp:** use an *exponential moving average* of $g^2$ instead → LR stays alive.
- **Adam:** momentum (1st moment) + RMSProp (2nd moment) + bias correction.
- **AdamW:** Adam but weight decay is applied to the weights directly, not folded
  into the gradient — better generalization.

## 3. Math derivation — every update rule

Let $g_t=\nabla_\theta\mathcal L(\theta_t)$.

**SGD.** $\theta_{t+1}=\theta_t-\eta\,g_t.$

**Momentum (heavy ball).** Velocity = EMA of gradients:
$$v_t=\mu v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta\,v_t.$$
Consistent directions accumulate ($\sum \mu^k$ amplifies by $\tfrac{1}{1-\mu}$);
oscillating ones cancel.

**Nesterov.** Evaluate the gradient at the *lookahead* $\tilde\theta=\theta_t-\eta\mu v_{t-1}$:
$$v_t=\mu v_{t-1}+\nabla\mathcal L(\tilde\theta),\qquad \theta_{t+1}=\theta_t-\eta v_t.$$
Looking ahead lets it brake *before* overshooting — a better-conditioned momentum.

**AdaGrad.** $G_t=G_{t-1}+g_t^2$ (elementwise), then
$$\theta_{t+1}=\theta_t-\frac{\eta}{\sqrt{G_t}+\epsilon}\odot g_t.$$
The per-coordinate denominator $\sqrt{G_t}$ shrinks steps for frequently-large grads.

**RMSProp.** Replace the growing sum with a decaying average:
$$E_t=\rho E_{t-1}+(1-\rho)g_t^2,\qquad
  \theta_{t+1}=\theta_t-\frac{\eta}{\sqrt{E_t}+\epsilon}\odot g_t.$$

**Adam.** Track both moments and *correct their bias*:
$$m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\qquad v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2.$$
Because $m_0=v_0=0$, unrolling gives $m_t=(1-\beta_1)\sum_{i=1}^{t}\beta_1^{t-i}g_i$.
Taking expectation under (approximately) stationary $g$:
$$\mathbb E[m_t]=(1-\beta_1^t)\,\mathbb E[g],\qquad \mathbb E[v_t]=(1-\beta_2^t)\,\mathbb E[g^2],$$
so the EMAs **under-estimate** the true moments by the factors $(1-\beta^t)$. Divide
them out:
$$\hat m_t=\frac{m_t}{1-\beta_1^{t}},\qquad \hat v_t=\frac{v_t}{1-\beta_2^{t}},
  \qquad \theta_{t+1}=\theta_t-\eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$
We verify in the demo that this restores an update of size $\approx\eta$ from step 1.

**AdamW (decoupled weight decay).** Standard L2 puts $\lambda\theta$ *inside* $g_t$,
so it gets divided by $\sqrt{\hat v_t}$ along with everything else (coupling the
regularization to the adaptive scale). AdamW instead decays the weights directly:
$$\theta_{t+1}=\theta_t-\eta\Big(\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}+\lambda\theta_t\Big).$$

## 4. NumPy implementation — each `step` is literally the update rule

In [ ]:
# ===== actual implementation from optimizers.py =====
from __future__ import annotations

import numpy as np

SEED = 0

QUAD_A, QUAD_B = 20.0, 1.0

def quad(p):
    x, y = p
    return 0.5 * (QUAD_A * x ** 2 + QUAD_B * y ** 2)

def quad_grad(p):
    x, y = p
    return np.array([QUAD_A * x, QUAD_B * y])

def beale(p):
    """Beale function: a classic non-convex 2D test, min 0 at (3, 0.5)."""
    x, y = p
    return ((1.5 - x + x * y) ** 2
            + (2.25 - x + x * y ** 2) ** 2
            + (2.625 - x + x * y ** 3) ** 2)

def beale_grad(p):
    """Analytic gradient of the Beale function (chain rule on each term)."""
    x, y = p
    a = 1.5 - x + x * y
    b = 2.25 - x + x * y ** 2
    c = 2.625 - x + x * y ** 3
    dx = 2 * a * (y - 1) + 2 * b * (y ** 2 - 1) + 2 * c * (y ** 3 - 1)
    dy = 2 * a * x + 2 * b * (2 * x * y) + 2 * c * (3 * x * y ** 2)
    return np.array([dx, dy])

class Optimizer:
    """Base: holds params (a single flat vector here) and applies an update."""

    name = "optimizer"

    def __init__(self, params: np.ndarray, lr: float):
        self.p = np.array(params, dtype=float)   # parameter vector theta
        self.lr = lr
        self.t = 0                               # timestep (for bias correction)

    def step(self, grad: np.ndarray) -> np.ndarray:
        raise NotImplementedError

def optimize(opt_cls, start, grad_fn, steps=400, **kw):
    """Run an optimizer on grad_fn from `start`, returning the trajectory.

    Nesterov is special: it needs the gradient evaluated at a lookahead point.
    """
    opt = opt_cls(np.array(start, dtype=float), **kw)
    traj = [opt.p.copy()]
    for _ in range(steps):
        if isinstance(opt, Nesterov):
            g = grad_fn(opt.lookahead())
        else:
            g = grad_fn(opt.p)
        opt.step(g)
        traj.append(opt.p.copy())
    return np.array(traj)

NUMPY_OPTS = {
    "sgd": (SGD, dict(lr=0.05)),
    "momentum": (Momentum, dict(lr=0.02, mu=0.9)),
    "nesterov": (Nesterov, dict(lr=0.02, mu=0.9)),
    "adagrad": (AdaGrad, dict(lr=0.5)),
    "rmsprop": (RMSProp, dict(lr=0.1, rho=0.9)),
    "adam": (Adam, dict(lr=0.1, b1=0.9, b2=0.999)),
    "adamw": (AdamW, dict(lr=0.1, wd=0.0)),
}

import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    start = (2.0, 2.0)
    target = np.array([0.0, 0.0])           # min of the quadratic bowl

    print(f"Ill-conditioned quadratic (cond = {QUAD_A/QUAD_B:.0f}) from (2, 2); "
          "min at (0, 0).")
    print(f"{'optimizer':10s} {'steps':>6s} {'f(final)':>12s} {'dist-to-min':>12s}")
    trajs = {}
    for nm, (cls, kw) in NUMPY_OPTS.items():
        traj = optimize(cls, start, quad_grad, steps=300, **kw)
        trajs[nm] = traj
        final = traj[-1]
        print(f"{nm:10s} {len(traj)-1:6d} {quad(final):12.4e} "
              f"{np.linalg.norm(final - target):12.4e}")

    # Bias-correction illustration. m,v are EMAs initialised at 0, so for small t
    # they are biased toward 0 -- and crucially v is biased MORE than m (b2 > b1),
    # so the ratio m/sqrt(v) is mis-scaled early. Feed a CONSTANT gradient g, whose
    # true moments are mean(g)=g and mean(g^2)=g^2: a correct estimator must give an
    # update of size exactly lr at every step. Watch the corrected version do that.
    print("\nAdam bias correction (constant gradient; ideal |update| = lr = 0.1):")
    print(f"  {'t':>3s} {'|update| uncorrected':>22s} {'|update| corrected':>20s}")
    b1, b2, lr, eps = 0.9, 0.999, 0.1, 1e-8
    m = np.zeros(2); v = np.zeros(2)
    g = np.array([1.0, -1.0])               # constant unit gradient
    for t in range(1, 6):
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        raw = lr * m / (np.sqrt(v) + eps)               # NO correction
        cor = lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)  # corrected
        per = np.abs(raw[0]); pec = np.abs(cor[0])      # symmetric coords
        print(f"  {t:3d} {per:22.4f} {pec:20.4f}")
    print("  -> uncorrected step is MIS-SCALED early (v under-estimated more than m,")
    print("     so m/sqrt(v) is off and only drifts toward lr slowly); the 1/(1-b^t)")
    print("     factors give the correct ~lr step from the very first iteration.")

    # cross-check NumPy vs torch.optim on the same problem. SGD/Momentum/Adam/AdamW
    # use identical formulas so they match to machine precision; torch's Nesterov &
    # the Adagrad/RMSprop eps-placement differ slightly, so we just report those.
    print("\nNumPy vs torch.optim final point (max coord diff over 300 steps):")
    for nm in NUMPY_OPTS:
        cls, kw = NUMPY_OPTS[nm]
        np_traj = optimize(cls, start, quad_grad, steps=300, **kw)
        t_traj = torch_optimize(nm, start, steps=300)
        diff = np.max(np.abs(np_traj[-1] - t_traj[-1]))
        print(f"  {nm:9s}: {diff:.2e}")
    print("  -> SGD/Momentum/Adam/AdamW match exactly (same update rule);")
    print("     small diffs for Nesterov/Adagrad/RMSProp = framework formulation details.")


class SGD(Optimizer):
    r"""Vanilla gradient descent:  theta <- theta - lr * g."""
    name = "sgd"

    def step(self, g):
        self.p -= self.lr * g
        return self.p


class Momentum(Optimizer):
    r"""Heavy ball: accumulate a velocity (EMA of gradients).
        v <- mu*v + g ;  theta <- theta - lr*v.
    The velocity averages out oscillation across steep directions and builds up
    speed along consistent (low-curvature) directions."""
    name = "momentum"

    def __init__(self, params, lr, mu=0.9):
        super().__init__(params, lr)
        self.mu = mu
        self.v = np.zeros_like(self.p)

    def step(self, g):
        self.v = self.mu * self.v + g
        self.p -= self.lr * self.v
        return self.p


class Nesterov(Optimizer):
    r"""Nesterov accelerated gradient: evaluate the gradient at the *lookahead*
    point theta - lr*mu*v (where momentum is about to carry us), giving a
    correction term. Equivalent reformulation used here:
        v       <- mu*v + g
        update  <- mu*v + g            (= mu*v_new + g, the lookahead-corrected step)
        theta   <- theta - lr*update.
    Note: the gradient g is supplied at the lookahead point by the caller."""
    name = "nesterov"

    def __init__(self, params, lr, mu=0.9):
        super().__init__(params, lr)
        self.mu = mu
        self.v = np.zeros_like(self.p)

    def lookahead(self):
        """Point at which the gradient should be evaluated for this step."""
        return self.p - self.lr * self.mu * self.v

    def step(self, g):
        # g is grad at the lookahead point self.lookahead()
        self.v = self.mu * self.v + g
        self.p -= self.lr * self.v
        return self.p


class AdaGrad(Optimizer):
    r"""Per-parameter learning rate that shrinks with accumulated gradient energy:
        G <- G + g^2 ;  theta <- theta - lr * g / (sqrt(G)+eps).
    Great for sparse features, but G only grows -> the effective LR decays to 0."""
    name = "adagrad"

    def __init__(self, params, lr, eps=1e-8):
        super().__init__(params, lr)
        self.eps = eps
        self.G = np.zeros_like(self.p)

    def step(self, g):
        self.G += g ** 2
        self.p -= self.lr * g / (np.sqrt(self.G) + self.eps)
        return self.p


class RMSProp(Optimizer):
    r"""Fix AdaGrad's monotonic decay by using an EMA of squared gradients:
        E <- rho*E + (1-rho)*g^2 ;  theta <- theta - lr * g / (sqrt(E)+eps).
    The window 'forgets' old gradients so the effective LR stays alive."""
    name = "rmsprop"

    def __init__(self, params, lr, rho=0.9, eps=1e-8):
        super().__init__(params, lr)
        self.rho, self.eps = rho, eps
        self.E = np.zeros_like(self.p)

    def step(self, g):
        self.E = self.rho * self.E + (1 - self.rho) * g ** 2
        self.p -= self.lr * g / (np.sqrt(self.E) + self.eps)
        return self.p


class Adam(Optimizer):
    r"""Adam = Momentum (1st moment m) + RMSProp (2nd moment v) + bias correction.
        m <- b1*m + (1-b1)*g                 (EMA of gradient)
        v <- b2*v + (1-b2)*g^2               (EMA of squared gradient)
        m_hat = m/(1-b1^t),  v_hat = v/(1-b2^t)   (correct the zero-init bias)
        theta <- theta - lr * m_hat / (sqrt(v_hat)+eps).
    Bias correction matters because m,v start at 0 and are badly under-estimated
    for small t; dividing by (1-b^t) -> 1 as t grows removes that startup bias."""
    name = "adam"

    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        super().__init__(params, lr)
        self.b1, self.b2, self.eps = b1, b2, eps
        self.m = np.zeros_like(self.p)
        self.v = np.zeros_like(self.p)

    def step(self, g):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g ** 2
        m_hat = self.m / (1 - self.b1 ** self.t)
        v_hat = self.v / (1 - self.b2 ** self.t)
        self.p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return self.p


class AdamW(Adam):
    r"""AdamW DECOUPLES weight decay from the gradient. Classic L2 adds wd*theta
    to the gradient (so it gets scaled by 1/sqrt(v) like everything else). AdamW
    instead applies the decay directly to the weights:
        theta <- theta - lr*( m_hat/(sqrt(v_hat)+eps) + wd*theta ).
    This makes the regularization strength independent of the adaptive scaling —
    the fix that made Adam competitive with SGD on generalization."""
    name = "adamw"

    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, wd=0.0):
        super().__init__(params, lr, b1, b2, eps)
        self.wd = wd

    def step(self, g):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g ** 2
        m_hat = self.m / (1 - self.b1 ** self.t)
        v_hat = self.v / (1 - self.b2 ** self.t)
        # decoupled decay: applied to theta, NOT routed through the adaptive term
        self.p -= self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.wd * self.p)
        return self.p

## 5. PyTorch implementation — same rules via `torch.optim` (validated)

In [ ]:
# ===== actual implementation from optimizers.py =====
def torch_optimize(name, start, steps=400):
    """Minimize the (autograd-differentiated) quadratic bowl with torch.optim,
    returning the trajectory. Mirrors NUMPY_OPTS hyper-parameters exactly, so the
    NumPy update rules can be validated against the framework."""
    dev = get_device()
    theta = torch.tensor(start, dtype=torch.float64, device=dev, requires_grad=True)
    builders = {
        "sgd": lambda: torch.optim.SGD([theta], lr=0.05),
        "momentum": lambda: torch.optim.SGD([theta], lr=0.02, momentum=0.9),
        "nesterov": lambda: torch.optim.SGD([theta], lr=0.02, momentum=0.9, nesterov=True),
        "adagrad": lambda: torch.optim.Adagrad([theta], lr=0.5),
        "rmsprop": lambda: torch.optim.RMSprop([theta], lr=0.1, alpha=0.9),
        "adam": lambda: torch.optim.Adam([theta], lr=0.1, betas=(0.9, 0.999)),
        "adamw": lambda: torch.optim.AdamW([theta], lr=0.1, weight_decay=0.0),
    }
    opt = builders[name]()
    traj = [theta.detach().cpu().numpy().copy()]
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.5 * (QUAD_A * theta[0] ** 2 + QUAD_B * theta[1] ** 2)
        loss.backward()
        opt.step()
        traj.append(theta.detach().cpu().numpy().copy())
    return np.array(traj)

## 6. Run — race on a quadratic, show bias correction, match torch

In [ ]:
demo()

## 7. Visualization — optimizer trajectories on a contour plot

The classic picture on the **Beale** function (a hard non-convex surface): watch
how momentum/Nesterov build speed and curve toward the minimum, while plain SGD
crawls. The black star is the global minimum $(3, 0.5)$.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import optimizers as M

start = (-2.0, 2.0)
# per-optimizer LRs that behave well on Beale's steep landscape
beale_cfg = {
    "sgd":      (M.SGD,      dict(lr=2e-4)),
    "momentum": (M.Momentum, dict(lr=1e-4, mu=0.9)),
    "nesterov": (M.Nesterov, dict(lr=1e-4, mu=0.9)),
    "rmsprop":  (M.RMSProp,  dict(lr=0.02, rho=0.9)),
    "adam":     (M.Adam,     dict(lr=0.3, b1=0.9, b2=0.999)),
}

# contour grid
xs = np.linspace(-3.5, 4.0, 300); ys = np.linspace(-1.5, 2.5, 300)
Xg, Yg = np.meshgrid(xs, ys)
Z = M.beale([Xg, Yg])

plt.figure(figsize=(8, 5.5))
plt.contour(Xg, Yg, np.log1p(Z), levels=30, cmap="viridis", alpha=.6)
for nm, (cls, kw) in beale_cfg.items():
    traj = M.optimize(cls, start, M.beale_grad, steps=4000, **kw)
    plt.plot(traj[:, 0], traj[:, 1], "-", lw=1.6, label=nm)
plt.scatter([3], [0.5], c="k", marker="*", s=180, zorder=5, label="min (3, 0.5)")
plt.scatter([start[0]], [start[1]], c="r", marker="o", s=40, zorder=5, label="start")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Optimizer trajectories on Beale (log-contours)")
plt.legend(loc="upper left", fontsize=8); plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Momentum** is almost free and fixes SGD's zig-zag on ill-conditioned losses.
- **AdaGrad** decays its LR to zero (bad for long training); **RMSProp** fixes that
  with an EMA.
- **Adam** is the default for most deep nets; **bias correction** is essential for
  sane early steps (we measured it).
- **AdamW > Adam + L2** when you care about generalization — decouple the decay.
- Adaptive methods can converge to *worse* minima than well-tuned SGD+momentum on
  some vision tasks; there is no universally best optimizer.
- LR scheduling/warmup composes with any of these — see the transformer file and
  `training-techniques/README.md`.